Here we combine the three recommendation system (the topic based recommender, ..., ... ) to make a combined recommendation system.

In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


First we connect to the database and get the events (ID, title, description) and the events users betted on.

In [37]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

Here we make the embeddings for the topic-based recommender.

In [38]:
import pandas as pd
import faiss
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pickle
import math

from utils import load_events_df, load_user_event_associations


index_path = "event_embeddings_shard_"
meta_path = "models/event_index_meta.pkl"
model_name_label = "all-MiniLM-L6-v2"

user_event_associations_df = await load_user_event_associations(conn)

print(f"Loaded {len(user_event_associations_df):,} user-event rows")

# get combined events with caching
events_df = await load_events_df(conn)
print(f"Loaded {len(events_df)} rows into events_df")

shard_paths = sorted(
    os.path.join("models", p)
    for p in os.listdir("models")
    if p.startswith(index_path)
)

have_shards = len(shard_paths) > 0

if have_shards and os.path.exists(meta_path):
    print("Loading FAISS index and metadata from disk...")

    # load FAISS index
    indices = []
    shard_paths = sorted(p for p in os.listdir("models") if p.startswith("event_embeddings_shard_"))

    for p in shard_paths:
        indices.append(faiss.read_index(os.path.join("models", p)))

    d = indices[0].d
    index = faiss.IndexFlatIP(d)

    def get_vectors(idx: faiss.Index) -> np.ndarray:
        """Return all vectors stored in a flat FAISS index as (n, d) float32 array."""
        n = idx.ntotal
        if hasattr(idx, "reconstruct_n"):
            return idx.reconstruct_n(0, n)
        # fallback: reconstruct one by one if reconstruct_n is missing
        return np.vstack(idx.reconstruct(i) for i in range(n))

    for shard_id, idx in enumerate(indices):
        xb = get_vectors(idx).astype("float32")
        index.add(xb)
        print(f"Added shard {shard_id} with {idx.ntotal} vectors")

    print("Merged index ntotal:", index.ntotal)

    # load mappings / metadata
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    index_to_id = meta["index_to_id"]
    id_to_index = meta["id_to_index"]
    model_name = meta.get("model_name", model_name_label)

    # reload embedding model
    embedding_model = SentenceTransformer(model_name)

    if hasattr(index, "reconstruct_n"):
        normalized_embeddings = index.reconstruct_n(0, index.ntotal)  # shape (n, d)
    else:
        # slower fallback
        normalized_embeddings = np.vstack(index.reconstruct(i) for i in range(n))

    print(normalized_embeddings.shape)
else:
    print("No saved index found. Building FAISS index from scratch...")
    topic_model = BERTopic(embedding_model=model_name_label)

    # Create combined text for embeddings
    events_df["combined_text"] = (
        events_df["title"].fillna("") + " " + events_df["description"].fillna("")
    )

    embedding_model = SentenceTransformer(model_name_label)
    embeddings = embedding_model.encode(
        events_df["combined_text"].tolist(), show_progress_bar=True
    )

    embedding_dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)

    normalized_embeddings = embeddings / np.linalg.norm(
        embeddings, axis=1, keepdims=True
    )
    index.add(normalized_embeddings.astype("float32"))

    index_to_id = {i: id for i, id in enumerate(events_df.index)}
    id_to_index = {id: i for i, id in enumerate(events_df.index)}

    # Save FAISS index
    emb = normalized_embeddings.astype("float32")
    n, d = emb.shape
    shard_size = 15000  # adjust to keep each file <100MB

    n_shards = math.ceil(n / shard_size)

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)
        part = emb[start:end]

        idx = faiss.IndexFlatIP(d)
        idx.add(part)

        faiss.write_index(idx, f"models/event_embeddings_shard_{shard_id:02d}.index")

    # Save mappings (and any other metadata you like)
    meta = {
        "index_to_id": index_to_id,
        "id_to_index": id_to_index,
        "model_name": model_name_label,
    }

    with open(meta_path, "wb") as f:
        pickle.dump(meta, f)

Loading user_event_associations_df from 14 cached files...
Loaded 483,767 user-event rows
Loading events_df from 3 cached parquet files in 'events_df_cache/'
Loaded 65056 rows into events_df
Loading FAISS index and metadata from disk...
Added shard 0 with 15000 vectors
Added shard 1 with 15000 vectors
Added shard 2 with 15000 vectors
Added shard 3 with 15000 vectors
Added shard 4 with 15000 vectors
Added shard 5 with 15000 vectors
Added shard 6 with 8450 vectors
Merged index ntotal: 98450
(98450, 384)


Here is our combined recommender:

In [39]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
from recommendation_system.main import get_recommendations_for_user

def combined_recommendation_for_user(user_address):
    # This is the topic based recommender
    top_n=10
    similar_per_event=5
    
    topic_based_recommendations= recommend_events_for_user_by_topic_based(
        id_to_index, 
        normalized_embeddings,
        index,
        index_to_id, 
        user_event_associations_df,
        events_df, 
        user_address, 
        top_n, 
        similar_per_event
    )
    association_based_recommendations = get_recommendations_for_user(user_address)
    return topic_based_recommendations, association_based_recommendations

In [40]:
topic_users = set(user_event_associations_df['address'].unique())

assoc_query = """
    SELECT DISTINCT up.address
    FROM "UserTrade" ut
    JOIN "UserProfile" up ON ut."proxyWallet" = up."proxyWallet"
    WHERE up.address IS NOT NULL
"""
assoc_rows = await conn.fetch(assoc_query)
assoc_users = set(row['address'] for row in assoc_rows)

combined = topic_users & assoc_users

print(f"Topic-based users:\t{len(topic_users):,}")
print(f"Association users:\t{len(assoc_users):,}")
print(f"Combined:\t{len(combined):,}")

Topic-based users:	24,640
Association users:	8,584
Combined:	8,584


In [41]:
example_user = list(combined)[0]
reccomandation_systems = combined_recommendation_for_user(example_user)

if reccomandation_systems:
    topic_rec = reccomandation_systems[0]
    association_rec = reccomandation_systems[1]
    
    print("TOPIC-BASED:")
    for rank, rec in enumerate(topic_rec, 1):
        print(f"{rank}. [{rec['id']}] {rec['title']}")

    print("ASSOCIATION RULES:")
    for rank, rec in enumerate(association_rec, 1):
        print(f"\n   {rank}. {rec['recommended_market']}")
        print(f"\tBased on: {rec['based_on']}")
        print(f"\tConfidence: {rec['confidence']:.1%} (probability)")
        print(f"\tLift: {rec['lift']:.2f}x (correlation strength)")
        print(f"\tSupport: {rec['support']:.1%} (frequency)")


Connected successfully!
Current time: 2025-12-01 17:10:02.839934+00:00
STEP 1: FETCHING TRANSACTION DATA

TRANSACTION STATISTICS:
	Total transactions: 1977
	Average items per transaction: 46.37
	Total unique items: 37550

Minimum support threshold: 0.01 (1.0%)
STEP 6: GENERATING ASSOCIATION RULES

Minimum confidence: 0.3 (30.0%)
Minimum lift: 1.1
APRIORI ANALYSIS SUMMARY

TRANSACTIONS:
	Total: 1977
	Avg items: 46.37

FREQUENT ITEM SETS:
	Total: 498
	1-item_sets: 394
	2-item_sets: 102
	3-item_sets: 2

ASSOCIATION RULES:
	Total: 115
	Avg confidence: 0.493
	Avg lift: 8.021
	Max lift: 24.477

TOP 50 RULES (by lift):
	1. over-1pt2b-committed-to-the-megaeth-public-sale-863_Yes → over-1b-committed-to-the-megaeth-public-sale_Yes
		Confidence: 61.9%, Lift: 24.48
	2. over-1b-committed-to-the-megaeth-public-sale_Yes → over-1pt2b-committed-to-the-megaeth-public-sale-863_Yes
		Confidence: 52.0%, Lift: 24.48
	3. over-1pt8b-committed-to-the-megaeth-public-sale-982-115-476-233-878_Yes → over-1pt4b-com